In [14]:
import pandas as pd
from pathlib import Path
import os

def select_scenario() -> Path:
    # Get all folder names from the logs folder
    logs_folder = Path(os.path.abspath('')) / 'logs'
    scenarios = [folder for folder in logs_folder.iterdir() if folder.is_dir()]
    print('Select a scenario:')
    for i, folder in enumerate(scenarios):
        print(f"[{i}]: {folder.name}")
    print("Your selection: ")
    selection = int(input())
    return scenarios[selection]

scenario_folder = select_scenario()

Select a scenario:
[0]: soil
Your selection: 


In [15]:
rx_packets_df_col_names = ["Time","Context","Packet UID","Channel freq","Channel no","Rate","Is short preamble","Sender node ID", "Receiver node ID", "Mode","Retries","Ness","Nss","Is short guard interval","Is stbc","Tx power level","Noise","Signal","Packets sent", "Packets received", "Packets dropped", "Packet retries", "Packet first seen", "Latency"]
rx_packets_df = pd.read_csv(scenario_folder / 'monitor_sniffer_rx.csv', delimiter=';', names=rx_packets_df_col_names, index_col=False)
rx_packets_df

,Time,Context,Packet UID,Channel freq,Channel no,Rate,Is short preamble,Sender node ID,Receiver node ID,Mode,...,Is stbc,Tx power level,Noise,Signal,Packets sent,Packets received,Packets dropped,Packet retries,Packet first seen,Latency
0,+2732007.0ns,/NodeList/112/DeviceList/0/$ns3::WifiNetDevice...,0,905,1,128,False,112,200,OfdmRate300KbpsBW1MHz,...,False,0,-113.976,-68.3716,1,0,0,0,+2732007.0ns,+0.0ns
1,+2732008.0ns,/NodeList/96/DeviceList/0/$ns3::WifiNetDevice/...,0,905,1,128,False,96,200,OfdmRate300KbpsBW1MHz,...,False,0,-113.976,-69.4508,1,1,0,0,+2732007.0ns,+1.0ns
2,+2732008.0ns,/NodeList/97/DeviceList/0/$ns3::WifiNetDevice/...,0,905,1,128,False,97,200,OfdmRate300KbpsBW1MHz,...,False,0,-113.976,-68.9425,1,2,0,0,+2732007.0ns,+1.0ns
3,+2732008.0ns,/NodeList/98/DeviceList/0/$ns3::WifiNetDevice/...,0,905,1,128,False,98,200,OfdmRate300KbpsBW1MHz,...,False,0,-113.976,-69.4508,1,3,0,0,+2732007.0ns,+1.0ns
4,+2732008.0ns,/NodeList/111/DeviceList/0/$ns3::WifiNetDevice...,0,905,1,128,False,111,200,OfdmRate300KbpsBW1MHz,...,False,0,-113.976,-68.9425,1,4,0,0,+2732007.0ns,+1.0ns
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4986063,+3599976920025.0ns,/NodeList/15/DeviceList/0/$ns3::WifiNetDevice/...,51217,905,1,128,False,15,200,OfdmRate300KbpsBW1MHz,...,False,0,-113.976,-79.7997,1,195,0,0,+3599976920007.0ns,+18.0ns
4986064,+3599976920025.0ns,/NodeList/29/DeviceList/0/$ns3::WifiNetDevice/...,51217,905,1,128,False,29,200,OfdmRate300KbpsBW1MHz,...,False,0,-113.976,-79.7997,1,196,0,0,+3599976920007.0ns,+18.0ns
4986065,+3599976920025.0ns,/NodeList/195/DeviceList/0/$ns3::WifiNetDevice...,51217,905,1,128,False,195,200,OfdmRate300KbpsBW1MHz,...,False,0,-113.976,-79.7997,1,197,0,0,+3599976920007.0ns,+18.0ns
4986066,+3599976920027.0ns,/NodeList/0/DeviceList/0/$ns3::WifiNetDevice/P...,51217,905,1,128,False,0,200,OfdmRate300KbpsBW1MHz,...,False,0,-113.976,-80.3893,1,198,0,0,+3599976920007.0ns,+20.0ns


In [16]:
rx_packets_df.drop_duplicates(subset=['Packet UID'], keep='last', inplace=True)

In [17]:
# Average the signal strength for each time
rx_packets_df_grouped_time = rx_packets_df.groupby('Time')
rx_packets_df_avg_signal = rx_packets_df_grouped_time['Signal'].mean()
rx_packets_df_avg_signal

Time
+1000040920027.0ns   -80.3893
+1000245720027.0ns   -80.3893
+1000450520027.0ns   -80.3893
+1000655320027.0ns   -80.3893
+1000860120027.0ns   -80.3893
                       ...   
+999221720027.0ns    -80.3893
+999426520027.0ns    -80.3893
+99945120027.0ns     -80.3893
+999631320027.0ns    -80.3893
+999836120027.0ns    -80.3893
Name: Signal, Length: 26845, dtype: float64

In [18]:
# Average the latency for each time
rx_packets_df['Latency'] = rx_packets_df['Latency'].str.replace('ns', '')
rx_packets_df['Latency'] = pd.to_numeric(rx_packets_df['Latency'])
rx_packets_df_grouped_latency = rx_packets_df.groupby('Latency')
rx_packets_df_avg_latency = rx_packets_df_grouped_time['Latency'].mean()
rx_packets_df_avg_latency

Time
+1000040920027.0ns    20.0
+1000245720027.0ns    20.0
+1000450520027.0ns    20.0
+1000655320027.0ns    20.0
+1000860120027.0ns    20.0
                      ... 
+999221720027.0ns     20.0
+999426520027.0ns     20.0
+99945120027.0ns      20.0
+999631320027.0ns     20.0
+999836120027.0ns     20.0
Name: Latency, Length: 26845, dtype: float64

In [19]:
#import matplotlib.pyplot as plt

#fig = plt.figure(figsize=(10, 5))
#ax = fig.add_subplot(111)
# Plot a graph where x = time, y = mean signal for that time
#ax.plot(rx_packets_df_avg_signal.index, rx_packets_df_avg_signal.values)
#plt.xlabel('Time (s)')
#plt.ylabel('Average RSSI across all nodes (dBm)')
#plt.show()

In [20]:
node_pos_df = pd.read_csv(scenario_folder / 'course_change.csv', delimiter=';', names=["Time","Context","Type","ID","x","y","z"], index_col=False)
node_pos_df.drop_duplicates(subset=["Time","ID"], keep='last', inplace=True)
node_pos_df.reset_index(drop=True, inplace=True)
node_pos_df

,Time,Context,Type,ID,x,y,z
0,+0.0ns,/NodeList/0/$ns3::MobilityModel/CourseChange,STA,0,0.4,0.4,-0.3
1,+0.0ns,/NodeList/1/$ns3::MobilityModel/CourseChange,STA,1,1.2,0.4,-0.3
2,+0.0ns,/NodeList/2/$ns3::MobilityModel/CourseChange,STA,2,2.0,0.4,-0.3
3,+0.0ns,/NodeList/3/$ns3::MobilityModel/CourseChange,STA,3,2.8,0.4,-0.3
4,+0.0ns,/NodeList/4/$ns3::MobilityModel/CourseChange,STA,4,3.6,0.4,-0.3
...,...,...,...,...,...,...,...
196,+0.0ns,/NodeList/196/$ns3::MobilityModel/CourseChange,STA,196,1.2,10.8,-0.3
197,+0.0ns,/NodeList/197/$ns3::MobilityModel/CourseChange,STA,197,2.0,10.8,-0.3
198,+0.0ns,/NodeList/198/$ns3::MobilityModel/CourseChange,STA,198,2.8,10.8,-0.3
199,+0.0ns,/NodeList/199/$ns3::MobilityModel/CourseChange,STA,199,3.6,10.8,-0.3


In [21]:
avg_signal_per_node = rx_packets_df.groupby('Sender node ID')['Signal'].mean()
worst_signal = avg_signal_per_node.max()

avg_signal_per_node_offset = avg_signal_per_node - worst_signal
avg_signal_per_node_offset

Sender node ID
0     -21.577100
1     -16.459394
2     -16.365471
3     -17.206106
4     -19.357182
         ...    
196   -18.463779
197   -17.085331
198   -17.578255
199   -18.034187
200   -23.650510
Name: Signal, Length: 199, dtype: float64

In [22]:
avg_latency_per_node = rx_packets_df.groupby('Sender node ID')['Latency'].mean()
best_latency = avg_latency_per_node.min()

avg_latency_per_node_offset = avg_latency_per_node - best_latency
avg_latency_per_node_offset

Sender node ID
0      1.247620e+07
1      1.399033e+08
2      1.021580e+08
3      9.303975e+10
4      7.802592e+07
           ...     
196    7.885394e+10
197    1.553783e+10
198    9.603610e+10
199    1.948576e+11
200    9.225914e+11
Name: Latency, Length: 199, dtype: float64

In [ ]:
import tkinter
import matplotlib
from matplotlib.backends.backend_tkagg import *
matplotlib.use("TkAgg")

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

root = tkinter.Tk()
root.wm_title("3D signal strength plot")

# Map 'Type' to colors
color_map = {'AP': 'red', 'STA': 'blue'}
colors = node_pos_df['Type'].map(color_map)

fig = plt.figure(dpi=100)

# Create four subplots
ax_node_pos = fig.add_subplot(221, projection='3d')
ax_signal = fig.add_subplot(222, projection='3d')
ax_latency = fig.add_subplot(223, projection='3d')
ax_unused = fig.add_subplot(224, projection='3d')

# Plot scatter on first subplot
ax_node_pos.scatter(node_pos_df['x'], node_pos_df['y'], node_pos_df['z'], c=colors)
ax_node_pos.set_title('Node positions')
ax_node_pos.set_xlabel('X (cartesian coord, m)')
ax_node_pos.set_ylabel('Y (cartesian coord, m)')
ax_node_pos.set_zlabel('Z (cartesian coord, m)')

# Plot bars on second subplot, where height is avg_signal_per_node_offset
for i, row in node_pos_df.iterrows():
    node_id = row['ID']
    if node_id in avg_signal_per_node_offset.index:
        delta = avg_signal_per_node_offset[node_id]
        # Ignore the outlier node (AP) when calculating color scale
        sta_values = avg_signal_per_node_offset[node_pos_df[
            (node_pos_df['Type'] == 'STA') & 
            (node_pos_df['ID'].isin(avg_signal_per_node_offset.index))
        ]['ID']]
        if not sta_values.empty:
            color = plt.cm.coolwarm((sta_values.max() - delta) / (sta_values.max() - sta_values.min()))
            ax_signal.bar3d(row['x'], row['y'], 0, .5, .5, delta, color=color)
ax_signal.set_title('Average RSSI')
ax_signal.set_xlabel('X (cartesian coord, m)')
ax_signal.set_ylabel('Y (cartesian coord, m)')
# Plot bars on third subplot, where height is avg_latency_per_node_offset
for i, row in node_pos_df.iterrows():
    node_id = row['ID']
    if node_id in avg_latency_per_node_offset.index:
        delta = avg_latency_per_node_offset[node_id]
        # Ignore the outlier node (AP) when calculating color scale
        sta_values = avg_signal_per_node_offset[node_pos_df[
            (node_pos_df['Type'] == 'STA') & 
            (node_pos_df['ID'].isin(avg_signal_per_node_offset.index))
        ]['ID']]
        if not sta_values.empty:
            color = plt.cm.coolwarm((sta_values.max() - delta) / (sta_values.max() - sta_values.min()))
            ax_latency.bar3d(row['x'], row['y'], 0, .5, .5, delta, color=color)
ax_latency.set_title('Average latency')
ax_latency.set_xlabel('X (cartesian coord, m)')
ax_latency.set_ylabel('Y (cartesian coord, m)')
ax_latency.set_zlabel(f'Latency (ns), offset by {round(best_latency, 2)}')

# Sync the view limits
ax_node_pos.set_xlim(node_pos_df['x'].min(), node_pos_df['x'].max())
ax_node_pos.set_ylim(node_pos_df['y'].min(), node_pos_df['y'].max())
ax_signal.set_xlim(node_pos_df['x'].min(), node_pos_df['x'].max())
ax_signal.set_ylim(node_pos_df['y'].min(), node_pos_df['y'].max())
ax_latency.set_xlim(node_pos_df['x'].min(), node_pos_df['x'].max())
ax_latency.set_ylim(node_pos_df['y'].min(), node_pos_df['y'].max())

# Create canvas and toolbar
canvas = FigureCanvasTkAgg(fig, master=root)
canvas.draw()
toolbar = NavigationToolbar2Tk(canvas, root)
toolbar.update()
canvas.get_tk_widget().pack(side=tkinter.TOP, fill=tkinter.BOTH, expand=1)

# Function to sync rotation
def on_rotate(event):
    axes = [ax_node_pos, ax_signal, ax_latency]
    if event.inaxes in axes:
        # Get the current elevation and azimuth from the active axis
        elev = event.inaxes.elev
        azim = event.inaxes.azim
        
        # Update all other axes except the active one and unused one
        for ax in axes:
            if ax != event.inaxes:
                ax.view_init(elev=elev, azim=azim)
        
        fig.canvas.draw_idle()

fig.canvas.mpl_connect('motion_notify_event', on_rotate)

tkinter.mainloop()